# HW2 — Practical regression with scikit-learn

Solution for Assignment 2 in `01_linear_regression__concepts.qmd`.

Dataset: `data/House_Rent_Dataset.csv` — Indian rental listings from Kaggle.
Target: `Rent` (continuous). Features: a mix of numeric and categorical.

**Outline.**
1. EDA: shape, types, column glossary, target distribution, feature distributions, multicollinearity, concrete look at the data problems we'll have to fix
2. Missing data: inject some NaN values and demonstrate the standard strategies
3. Feature engineering (parse `Floor` and `Posted On`)
4. Encoding: dummy-trap demo, then manual `StandardScaler` / `OneHotEncoder` / `OrdinalEncoder` on individual columns so the pieces are clear
5. Wire the same pieces into a `Pipeline` + `ColumnTransformer` and fit
6. Coefficient interpretation + concrete prediction

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.compose import ColumnTransformer # scikit-learn scienfit toolkit
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

ARM_RED = "#D90012"; ARM_BLUE = "#0033A0"; ARM_ORANGE = "#F2A800"
SEED = 509
np.random.seed(SEED)

pio.renderers.default = "notebook_connected" 

## 1. EDA

In [2]:
df = pd.read_csv("data/House_Rent_Dataset.csv")
print("shape:", df.shape)
df.head()

shape: (4746, 12)


,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
0,2022-05-18,2,10000,1100,Ground out of 2,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner
1,2022-05-13,2,20000,800,1 out of 3,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
2,2022-05-16,2,17000,1000,1 out of 3,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
3,2022-07-04,2,10000,800,1 out of 2,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner
4,2022-05-09,2,7500,850,1 out of 2,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner


### Column glossary (from the Kaggle dataset description)

| Column | Description |
|---|---|
| `BHK` | Number of Bedrooms, Hall, Kitchen. |
| `Rent` | Rent of the Houses/Apartments/Flats. **(target)** |
| `Size` | Size of the Houses/Apartments/Flats in Square Feet. |
| `Floor` | Floor of the house and total number of floors in the building (e.g. "Ground out of 2", "3 out of 5"). |
| `Area Type` | Whether the size is reported as Super Area, Carpet Area, or Built Area. |
| `Area Locality` | Locality of the listing. |
| `City` | City where the listing is located. |
| `Furnishing Status` | Furnished, Semi-Furnished, or Unfurnished. |
| `Tenant Preferred` | Type of tenant the owner / agent prefers. |
| `Bathroom` | Number of bathrooms. |
| `Point of Contact` | Whom to contact for more information (Owner / Agent / Builder). |

In [3]:
df.dtypes

Posted On            object
BHK                   int64
Rent                  int64
Size                  int64
Floor                object
Area Type            object
Area Locality        object
City                 object
Furnishing Status    object
Tenant Preferred     object
Bathroom              int64
Point of Contact     object
dtype: object

In [4]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Posted On,4746,81,2022-07-06,311,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BHK,4746.0,NaN,NaN,NaN,2.08386,0.832256,1.0,2.0,2.0,3.0,6.0
Rent,4746.0,NaN,NaN,NaN,34993.451327,78106.412937,1200.0,10000.0,16000.0,33000.0,3500000.0
Size,4746.0,NaN,NaN,NaN,967.490729,634.202328,10.0,550.0,850.0,1200.0,8000.0
Floor,4746,480,1 out of 2,379,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Area Type,4746,3,Super Area,2446,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Area Locality,4746,2235,Bandra West,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,4746,6,Mumbai,972,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Furnishing Status,4746,3,Semi-Furnished,2251,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tenant Preferred,4746,3,Bachelors/Family,3444,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Target distribution

`Rent` ranges from 1.2k to 3.5M — heavy right skew. Median ~16k, mean ~35k. Linear regression on raw Rent will be driven by the extreme tail. We'll **log-transform the target**.

In [5]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Rent — raw", "Rent — log1p"))
fig.add_trace(go.Histogram(x=df["Rent"], nbinsx=60,
                           marker=dict(color=ARM_BLUE, opacity=0.85,
                                       line=dict(color="white", width=1)),
                           showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=np.log1p(df["Rent"]), nbinsx=60,
                           marker=dict(color=ARM_RED, opacity=0.85,
                                       line=dict(color="white", width=1)),
                           showlegend=False), row=1, col=2)
fig.update_xaxes(title="Rent", row=1, col=1)
fig.update_xaxes(title="log(1 + Rent)", row=1, col=2)
fig.update_yaxes(title="count")
fig.update_layout(width=1000, height=420,
                  title="Target distribution before and after log1p")
fig

After `log1p`, the target is roughly Gaussian. That fits linear regression much better.

**Note on what we're optimizing.** Mean squared error on `log(Rent)` is roughly equivalent to penalizing *relative* errors on raw Rent. A 5,000 flat predicted at 6,000 (20% off) is weighted the same as a 50,000 flat predicted at 60,000 (20% off). That's usually what we want for prices.

### Feature distributions

In [6]:
numeric_cols_raw = ["BHK", "Size", "Bathroom"]
fig = make_subplots(rows=1, cols=3, subplot_titles=numeric_cols_raw)
for j, c in enumerate(numeric_cols_raw):
    fig.add_trace(go.Histogram(x=df[c], nbinsx=30,
                               marker=dict(color=ARM_BLUE, opacity=0.85,
                                           line=dict(color="white", width=1)),
                               showlegend=False), row=1, col=j + 1)
fig.update_yaxes(title="count")
fig.update_layout(width=1100, height=380,
                  title="Numeric feature distributions")
fig

In [7]:
print("Unique counts of categorical-like columns:")
for c in ["Floor", "Area Type", "Area Locality", "City", "Furnishing Status",
          "Tenant Preferred", "Point of Contact", "Posted On"]:
    print(f"  {c:<22}: {df[c].nunique()} unique")

Unique counts of categorical-like columns:
  Floor                 : 480 unique
  Area Type             : 3 unique
  Area Locality         : 2235 unique
  City                  : 6 unique
  Furnishing Status     : 3 unique
  Tenant Preferred      : 3 unique
  Point of Contact      : 3 unique
  Posted On             : 81 unique


### Look at the actual problem cases

Before we describe the issues in words, let's just look at the data and see them for ourselves.

**Problem 1.** `Floor` is structured text, not a number. The information is "which floor / out of how many" — useful, but a linear model can't consume "Ground out of 2" directly.

In [8]:
print("Sample of unparsed Floor values:")
print(df["Floor"].head(15).tolist())
print()
print("Top-10 most common Floor strings (out of", df["Floor"].nunique(), "unique):")
print(df["Floor"].value_counts().head(10))

Sample of unparsed Floor values:
['Ground out of 2', '1 out of 3', '1 out of 3', '1 out of 2', '1 out of 2', 'Ground out of 1', 'Ground out of 4', '1 out of 2', '1 out of 2', '1 out of 3', '1 out of 4', '1 out of 1', '1 out of 4', '1 out of 2', 'Ground out of 2']

Top-10 most common Floor strings (out of 480 unique):
Floor
1 out of 2         379
Ground out of 2    350
2 out of 3         312
2 out of 4         308
1 out of 3         293
3 out of 4         239
Ground out of 3    209
1 out of 4         200
Ground out of 1    195
1 out of 1         134
Name: count, dtype: int64


**Problem 2.** `Point of Contact` has 3 categories, but one of them has exactly one row. After one-hot encoding it becomes a binary column that's almost always zero — pure noise that the model will fit to whatever single Rent value came with it.

In [9]:
print("Point of Contact — value counts:")
print(df["Point of Contact"].value_counts())
print()
print("The 1 row with 'Contact Builder':")
print(df[df["Point of Contact"] == "Contact Builder"][["City", "Rent", "Size", "Point of Contact"]])

Point of Contact — value counts:
Point of Contact
Contact Owner      3216
Contact Agent      1529
Contact Builder       1
Name: count, dtype: int64

The 1 row with 'Contact Builder':
           City  Rent  Size Point of Contact
4061  Hyderabad  5500   400  Contact Builder


**Problem 3.** `Area Locality` has 2,235 unique values across 4,746 rows — about 2 rows per locality. One-hot would create 2,235 columns, the vast majority of which would have a single 1 in them — useless for learning.

In [10]:
print(f"Area Locality unique: {df['Area Locality'].nunique()}")
print(f"Total rows:           {len(df)}")
print(f"Average rows / locality: {len(df) / df['Area Locality'].nunique():.2f}")
print()
print("Even the most common localities have very few rows each:")
print(df["Area Locality"].value_counts().head(10))

Area Locality unique: 2235
Total rows:           4746
Average rows / locality: 2.12

Even the most common localities have very few rows each:
Area Locality
Bandra West        37
Gachibowli         29
Electronic City    24
Velachery          22
Miyapur, NH 9      22
Madipakkam         20
Chembur            19
Laxmi Nagar        19
K R Puram          19
Kondapur           18
Name: count, dtype: int64


**Problem 4.** `Posted On` is a date string. Raw datetime strings carry useful signal (month-of-year, time-since-first-post) but aren't directly usable by a linear model.

In [11]:
print("Posted On sample:")
print(df["Posted On"].head(5).tolist())
print(f"\ndtype: {df['Posted On'].dtype}")

Posted On sample:
['2022-05-18', '2022-05-13', '2022-05-16', '2022-07-04', '2022-05-09']

dtype: object


**Recap of the cleanups we'll do.**

- `Floor` → parse into two numbers: `floor_number` and `total_floors`
- `Point of Contact` → merge "Contact Builder" into the most similar non-singleton class
- `Area Locality` → drop (too sparse for a baseline)
- `Posted On` → parse into `days_since_first_post` and `posted_month`
- `Rent` (target) → `log1p` (already discussed)

The other two issues we'll deal with shortly: missing data (which we'll inject) and multicollinearity (which we measure next).

### Multicollinearity check

`BHK`, `Size`, `Bathroom` are intuitively correlated (bigger flats have more rooms AND more bathrooms). Let's measure.

In [12]:
corr = df[["BHK", "Size", "Bathroom"]].corr()
print(corr.round(2))

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.columns,
    zmin=-1, zmax=1, colorscale="RdBu_r",
    text=corr.round(2).values, texttemplate="%{text}",
    textfont=dict(size=14, color="black"),
    colorbar=dict(title="corr"),
))
fig.update_layout(title="Pairwise correlations (BHK, Size, Bathroom)",
                  width=500, height=420,
                  yaxis=dict(autorange="reversed"))
fig

           BHK  Size  Bathroom
BHK       1.00  0.72      0.79
Size      0.72  1.00      0.74
Bathroom  0.79  0.74      1.00


`BHK` and `Bathroom` are highly correlated (~0.79), and `Size` is well-correlated with both (~0.72–0.74). This is **multicollinearity** — when features carry overlapping information, the linear regression's coefficient estimates become unstable (the model can shuffle "credit" between the correlated features almost arbitrarily). We won't fix it here (Ridge regression in the next chapter is the standard remedy), but it's important to recognize: when you see coefficient signs flip as you add features, multicollinearity is usually why.

## 2. Missing data

The raw CSV has no missing values, which is actually unusual for a real listings dataset. To get hands-on practice with the standard handling strategies, we'll **deliberately inject NaN values** into a few columns at known rates and then work through each option.

In [15]:
# Inject NaN to simulate a realistic missingness pattern
rng_na = np.random.default_rng(SEED)
n = len(df)

# 5% missing on Furnishing Status, 3% on Size, 2% on Floor
for col, rate in [("Furnishing Status", 0.05), ("Size", 0.03), ("Floor", 0.02)]:
    idx = rng_na.choice(n, size=int(n * rate), replace=False)
    df.loc[idx, col] = np.nan

print("NaN per column after injection:")
print(df.isna().sum()[df.isna().sum() > 0])

NaN per column after injection:
Size                 142
Floor                 94
Furnishing Status    237
dtype: int64


### Strategy 1 — drop rows with any NaN

The simplest fix: throw away every row that has any missing values. Quick to implement, but you lose information.

In [16]:
df.isna().sum()

Posted On              0
BHK                    0
Rent                   0
Size                 142
Floor                 94
Area Type              0
Area Locality          0
City                   0
Furnishing Status    237
Tenant Preferred       0
Bathroom               0
Point of Contact       0
dtype: int64

In [17]:
n_before = len(df)
df_dropped = df.dropna()
n_after = len(df_dropped)
print(f"rows before drop: {n_before:,}")
print(f"rows after drop:  {n_after:,}")
print(f"rows lost:        {n_before - n_after:,}  ({(n_before - n_after) / n_before:.1%})")

rows before drop: 4,746
rows after drop:  4,283
rows lost:        463  (9.8%)


Dropping costs us about 10% of the data. Acceptable when missingness is rare and random; risky when missingness might be informative (e.g., listings without `Furnishing Status` might tend to be unfurnished). We'll do better.

### Strategy 2 — simple imputation

Fill numeric columns with the **median** (robust to outliers), categorical columns with the **mode** (most common value). Cheap, preserves all rows, but shrinks the variance of the imputed column and loses the signal that a value was missing in the first place.

In [18]:
# Numeric: Size gets filled with the median
size_median = df["Size"].median()
print(f"Size median: {size_median}")
print(f"NaN in Size before: {df['Size'].isna().sum()}")

# Categorical: Furnishing Status gets filled with the mode
furnishing_mode = df["Furnishing Status"].mode().iloc[0]
print(f"\nFurnishing Status mode: '{furnishing_mode}'")
print(f"NaN in Furnishing Status before: {df['Furnishing Status'].isna().sum()}")

Size median: 850.0
NaN in Size before: 142

Furnishing Status mode: 'Semi-Furnished'
NaN in Furnishing Status before: 237


### Strategy 3 — missing indicator + impute (the one we'll use)

Add a binary `Furnishing_was_missing` column alongside the imputed value. The model can now learn that "missing" itself carries signal — listings without furnishing info might tend to be unfurnished. We do this for `Furnishing Status` since the missingness might be informative; for `Size` we just median-impute (less likely to be informative).

In [19]:
# Indicator + impute for Furnishing Status (potentially informative missingness)
df["Furnishing_was_missing"] = df["Furnishing Status"].isna().astype(int)
df["Furnishing Status"] = df["Furnishing Status"].fillna(furnishing_mode)

# Plain median impute for Size
df["Size"] = df["Size"].fillna(size_median)

# Floor — we'll parse this into two numeric columns shortly; for the few rows
# where Floor is missing, the cheapest fix is to drop them (only 2%).
df = df.dropna(subset=["Floor"]).reset_index(drop=True)

print("NaN after handling:")
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().sum().sum() else "  (none)")
print(f"\nRows kept: {len(df):,}")
print(f"New column: Furnishing_was_missing")
print(df["Furnishing_was_missing"].value_counts().to_dict())

NaN after handling:
  (none)

Rows kept: 4,652
New column: Furnishing_was_missing
{0: 4416, 1: 236}


## Handle the singleton "Contact Builder"

We saw above that `Point of Contact = "Contact Builder"` shows up in exactly 1 row. Merge it into the nearest non-singleton class so we don't waste a coefficient on noise.

In [20]:
print("Before merge:", df["Point of Contact"].value_counts().to_dict())
df["Point of Contact"] = df["Point of Contact"].replace({"Contact Builder": "Contact Agent"})
print("After merge: ", df["Point of Contact"].value_counts().to_dict())

Before merge: {'Contact Owner': 3149, 'Contact Agent': 1502, 'Contact Builder': 1}
After merge:  {'Contact Owner': 3149, 'Contact Agent': 1503}


## 3. Feature engineering — parse `Floor` and `Posted On`

These columns hold real numeric information disguised as text. We can do better than label-encoding 480 floor strings.

In [21]:
def parse_floor(s):
    # Examples: "2 out of 4", "Ground out of 3", "Upper Basement out of 1"
    parts = str(s).lower().split(" out of ")
    if len(parts) != 2:
        return pd.Series([np.nan, np.nan])
    floor_str, total_str = parts
    if floor_str in ("ground",):
        floor = 0
    elif "basement" in floor_str:
        floor = -1
    else:
        try:
            floor = int(floor_str)
        except ValueError:
            floor = np.nan
    try:
        total = int(total_str)
    except ValueError:
        total = np.nan
    return pd.Series([floor, total])


df[["floor_number", "total_floors"]] = df["Floor"].apply(parse_floor)
print("Parsed floor sanity check:")
print(df[["Floor", "floor_number", "total_floors"]].head(8))
print("\nNaN after parsing:", df[["floor_number", "total_floors"]].isna().sum().to_dict())

Parsed floor sanity check:
             Floor  floor_number  total_floors
0  Ground out of 2           0.0           2.0
1       1 out of 3           1.0           3.0
2       1 out of 3           1.0           3.0
3       1 out of 2           1.0           2.0
4       1 out of 2           1.0           2.0
5  Ground out of 1           0.0           1.0
6  Ground out of 4           0.0           4.0
7       1 out of 2           1.0           2.0

NaN after parsing: {'floor_number': 4, 'total_floors': 4}


In [ ]:
# A few rows had floor strings we couldn't parse cleanly; median-impute them.
for c in ["floor_number", "total_floors"]:
    df[c] = df[c].fillna(df[c].median())


Date range: 2022-04-13 00:00:00 to 2022-07-11 00:00:00


In [23]:

df["Posted On"] = pd.to_datetime(df["Posted On"])
ref = df["Posted On"].min()
df["days_since_first_post"] = (df["Posted On"] - ref).dt.days
df["posted_month"] = df["Posted On"].dt.month
print("Date range:", df["Posted On"].min(), "to", df["Posted On"].max())

Date range: 2022-04-13 00:00:00 to 2022-07-11 00:00:00


## 4. Encoding categorical features + manual preprocessing

| Column | Encoding | Why |
|---|---|---|
| `Furnishing Status` | **Ordinal** | Genuinely ordered: Unfurnished < Semi-Furnished < Furnished |
| `Area Type` | One-hot | 3 unordered levels (`Super Area`, `Carpet Area`, `Built Area`) |
| `City` | One-hot | 6 unordered cities |
| `Tenant Preferred` | One-hot | 3 unordered preferences |
| `Point of Contact` | One-hot | 2 levels after we merged the singleton |
| `Area Locality` | **Drop** | 2,235 unique, too sparse for OHE |

### Aside — the "dummy trap" and why we use `drop="first"`

One-hot encoding a $k$-level categorical creates $k$ binary columns that **sum to 1 in every row**. If we also include an intercept (a column of 1s), those $k+1$ columns are linearly dependent — the intercept equals the sum of the OHE columns. That makes $X^\top X$ **singular**, and the normal equation literally cannot be solved.

In [25]:
# Toy categorical: 3 cities, 5 rows
city = pd.DataFrame({"city": ["Mumbai", "Chennai", "Bangalore", "Mumbai", "Chennai"]})

# OHE WITHOUT drop_first - all 3 levels become columns
ohe_full = OneHotEncoder(sparse_output=False, drop=None).fit_transform(city)
intercept = np.ones((len(city), 1))
X_bad = np.hstack([intercept, ohe_full])

print("X_bad (intercept + 3 OHE columns):")
print(X_bad)
print(f"\nshape:   {X_bad.shape}  (n=5, p=4)")
print(f"rank:    {np.linalg.matrix_rank(X_bad)}  <- expected 4, but is 3")
print("=> X^T X is singular. The normal equation cannot solve this:\n")

y_fake = np.array([10.0, 20.0, 30.0, 15.0, 25.0])
try:
    np.linalg.solve(X_bad.T @ X_bad, X_bad.T @ y_fake)
except np.linalg.LinAlgError as e:
    print(f"  LinAlgError: {e}")

X_bad (intercept + 3 OHE columns):
[[1. 0. 0. 1.]
 [1. 0. 1. 0.]
 [1. 1. 0. 0.]
 [1. 0. 0. 1.]
 [1. 0. 1. 0.]]

shape:   (5, 4)  (n=5, p=4)
rank:    3  <- expected 4, but is 3
=> X^T X is singular. The normal equation cannot solve this:

  LinAlgError: Singular matrix


**The fix.** Drop one OHE column (`drop_first=True` or `drop="first"`) — the dropped category becomes the implicit baseline, and the remaining $k - 1$ columns are linearly independent. We'll use this below.

### Doing the preprocessing manually — what each transformer actually does

sklearn's `StandardScaler`, `OneHotEncoder`, and `OrdinalEncoder` aren't magic — each one fits a few numbers and applies a formula. Let's apply them by hand on individual columns first, so the pipeline that comes next is just bookkeeping.

In [26]:
# === StandardScaler on Size ===
# Fits the mean and standard deviation; transforms via (x - mean) / std.
size_scaler = StandardScaler()
size_scaled = size_scaler.fit_transform(df[["Size"]])  # 2-D input expected

print("StandardScaler on Size:")
print(f"  fitted mean = {size_scaler.mean_[0]:.2f}")
print(f"  fitted std  = {size_scaler.scale_[0]:.2f}")
print(f"  raw Size head:    {df['Size'].head(5).tolist()}")
print(f"  scaled Size head: {size_scaled[:5].ravel().round(3).tolist()}")
print(f"  scaled mean      = {size_scaled.mean():.6f}  (should be 0)")
print(f"  scaled std       = {size_scaled.std():.6f}  (should be 1)")

StandardScaler on Size:
  fitted mean = 964.94
  fitted std  = 625.65
  raw Size head:    [1100.0, 800.0, 1000.0, 800.0, 850.0]
  scaled Size head: [0.216, -0.264, 0.056, -0.264, -0.184]
  scaled mean      = 0.000000  (should be 0)
  scaled std       = 1.000000  (should be 1)


In [27]:
# === OneHotEncoder on City ===
# Fits the list of unique categories; transforms each row to a 0/1 vector.
city_ohe = OneHotEncoder(sparse_output=False, drop="first")  # drop="first" = no dummy trap
city_encoded = city_ohe.fit_transform(df[["City"]])

print("OneHotEncoder on City:")
print(f"  categories found: {city_ohe.categories_[0].tolist()}")
print(f"  drop='first' removes: '{city_ohe.categories_[0][0]}' (it becomes the baseline)")
print(f"  output column names: {city_ohe.get_feature_names_out(['City']).tolist()}")
print(f"  shape of encoded matrix: {city_encoded.shape}")
print(f"  first 3 rows:")
print(city_encoded[:3])
print(f"  corresponding cities: {df['City'].head(3).tolist()}")

OneHotEncoder on City:
  categories found: ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai']
  drop='first' removes: 'Bangalore' (it becomes the baseline)
  output column names: ['City_Chennai', 'City_Delhi', 'City_Hyderabad', 'City_Kolkata', 'City_Mumbai']
  shape of encoded matrix: (4652, 5)
  first 3 rows:
[[0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0.]]
  corresponding cities: ['Kolkata', 'Kolkata', 'Kolkata']


In [28]:
# === OrdinalEncoder on Furnishing Status ===
# Maps each category to an integer in a user-specified order.
# We provide the order explicitly so it reflects the real meaning.
furn_enc = OrdinalEncoder(categories=[["Unfurnished", "Semi-Furnished", "Furnished"]])
furn_encoded = furn_enc.fit_transform(df[["Furnishing Status"]])

print("OrdinalEncoder on Furnishing Status:")
print(f"  categories (in our specified order): {furn_enc.categories_[0].tolist()}")
print(f"  Unfurnished -> 0, Semi-Furnished -> 1, Furnished -> 2")
print(f"  first 5 raw values:     {df['Furnishing Status'].head(5).tolist()}")
print(f"  first 5 encoded values: {furn_encoded[:5].ravel().tolist()}")

OrdinalEncoder on Furnishing Status:
  categories (in our specified order): ['Unfurnished', 'Semi-Furnished', 'Furnished']
  Unfurnished -> 0, Semi-Furnished -> 1, Furnished -> 2
  first 5 raw values:     ['Unfurnished', 'Semi-Furnished', 'Semi-Furnished', 'Unfurnished', 'Unfurnished']
  first 5 encoded values: [0.0, 1.0, 1.0, 0.0, 0.0]


Each transformer is small and self-contained:

- `StandardScaler` stores 2 numbers per column (mean, std).
- `OneHotEncoder` stores the list of categories and emits 0/1 columns.
- `OrdinalEncoder` stores the ordered category list and emits integers.

Now we'd like to apply different transformers to different columns and stitch the outputs back together. Writing that by hand is tedious. That's exactly what **`ColumnTransformer`** does.

## 5. Wire everything into a `Pipeline` + `ColumnTransformer`, fit on the full data

In [29]:
numeric_features = ["BHK", "Size", "Bathroom",
                    "floor_number", "total_floors",
                    "days_since_first_post", "posted_month",
                    "Furnishing_was_missing"]
onehot_features = ["Area Type", "City", "Tenant Preferred", "Point of Contact"]
ordinal_features = ["Furnishing Status"]
ordinal_levels = [["Unfurnished", "Semi-Furnished", "Furnished"]]

features_used = numeric_features + onehot_features + ordinal_features
print(f"Using {len(features_used)} columns + 1 target")

Using 13 columns + 1 target


In [30]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("ord", OrdinalEncoder(categories=ordinal_levels), ordinal_features),
        ("ohe", OneHotEncoder(handle_unknown="ignore", drop="first"), onehot_features),
    ],
    remainder="drop",
)

pipe = Pipeline([
    ("prep", preprocess),
    ("lr", LinearRegression()),
])

X = df[features_used]
y = np.log1p(df["Rent"])  # log-transformed target

pipe.fit(X, y)

# Empirical risk from HW1: mean squared error on log-Rent.
yhat = pipe.predict(X)
mse = float(np.mean((yhat - y) ** 2))
print(f"fitted pipeline. empirical risk (MSE on log-Rent): {mse:.4f}")

fitted pipeline. empirical risk (MSE on log-Rent): 0.1641


## 6. Coefficient interpretation

Because every feature was standardized (numeric → unit variance, OHE → 0/1, ordinal → 0/1/2), the magnitude of each coefficient is roughly comparable as "log-Rent units per standardized feature unit".

In [31]:
prep_fitted = pipe.named_steps["prep"]
feature_names = list(prep_fitted.get_feature_names_out())
coefs = pipe.named_steps["lr"].coef_
intercept = pipe.named_steps["lr"].intercept_

coef_df = (
    pd.DataFrame({"feature": feature_names, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
    .reset_index(drop=True)
)
print(f"intercept (log-Rent at zero scaled features): {intercept:.3f}")
coef_df

intercept (log-Rent at zero scaled features): 9.828


,feature,coef,abs_coef
0,ohe__City_Mumbai,0.907697,0.907697
1,ohe__Point of Contact_Contact Owner,-0.351535,0.351535
2,ohe__City_Kolkata,-0.282249,0.282249
3,num__Size,0.249551,0.249551
4,num__BHK,0.191912,0.191912
5,ohe__City_Delhi,0.186092,0.186092
6,ohe__City_Hyderabad,-0.150128,0.150128
7,num__Bathroom,0.141947,0.141947
8,ord__Furnishing Status,0.135791,0.135791
9,ohe__Tenant Preferred_Family,-0.128452,0.128452


In [27]:
order = coef_df.sort_values("coef")
colors = [ARM_RED if v < 0 else ARM_BLUE for v in order["coef"]]
fig = go.Figure()
fig.add_trace(go.Bar(x=order["coef"], y=order["feature"], orientation="h",
                     marker=dict(color=colors, opacity=0.85),
                     showlegend=False))
fig.add_vline(x=0, line=dict(color="black", width=1))
fig.update_xaxes(title="standardized coefficient (effect on log-Rent)")
fig.update_layout(title="Linear regression coefficients, sorted by sign",
                  width=800, height=620,
                  margin=dict(l=220))
fig

**What this tells us — and whether it makes sense.**

- The biggest **positive** drivers are usually `Size`, `Bathroom`, `BHK`, `Furnishing Status` — bigger / better-equipped flats cost more. Sanity check: passes.
- The biggest **negative** drivers are usually the city dummies for cheaper cities (since Mumbai is the OHE baseline after `drop="first"`). A negative coefficient on `City_Chennai` does NOT mean Chennai is cheap in absolute terms — it means Chennai is cheaper than the dropped reference category.
- Multicollinearity between `Size`, `BHK`, `Bathroom` means the *individual* coefficients are unstable — the model splits credit between them in a way that's noisy. The **sum** of their effects is reliable; individual effects are not.

**Big takeaway.** Coefficient interpretation only works cleanly when features are scaled AND the reference category for OHE is sensible. If you ever see "Chennai is cheap" without remembering the baseline city, you're misreading the model.

### Use the model — predict the rent of a specific house

In [32]:
new_house = pd.DataFrame([{
    "BHK": 2, "Size": 1000, "Bathroom": 2,
    "floor_number": 3, "total_floors": 5,
    "days_since_first_post": 30, "posted_month": 5,
    "Furnishing_was_missing": 0,
    "Area Type": "Super Area", "City": "Mumbai",
    "Tenant Preferred": "Bachelors/Family",
    "Point of Contact": "Contact Owner",
    "Furnishing Status": "Semi-Furnished",
}])

log_rent_pred = pipe.predict(new_house)[0]
rent_pred = np.expm1(log_rent_pred)
print(f"Hypothetical: 2BHK / 1000 sq ft / Mumbai / Semi-Furnished / 3rd floor")
print(f"  predicted log-Rent: {log_rent_pred:.2f}")
print(f"  predicted Rent:     Rs {rent_pred:,.0f}/month")

Hypothetical: 2BHK / 1000 sq ft / Mumbai / Semi-Furnished / 3rd floor
  predicted log-Rent: 10.54
  predicted Rent:     Rs 37,610/month


## Recap

- EDA caught: heavy Rent skew → `log1p`; `Floor` and `Posted On` as text → parsed into numbers; `Area Locality` too sparse → dropped; `Point of Contact = Contact Builder` singleton → merged. We looked at each problem in the actual data before describing it.
- Spotted multicollinearity between `BHK`, `Size`, `Bathroom` — coefficient signs on individual columns will be noisy as a result.
- Injected NaN into 3 columns to practice missing-data handling. Worked through 3 strategies (drop, simple impute, missing-indicator + impute) and used the indicator trick for `Furnishing Status` where the missingness may carry signal.
- Showed the **dummy trap** as a standalone demo, then applied `StandardScaler`, `OneHotEncoder`, `OrdinalEncoder` by hand on individual columns to see what each one actually does.
- Wired the same pieces into a `Pipeline` + `ColumnTransformer` and fit on the full data.
- Coefficient interpretation requires scaled features AND care with the OHE reference category.
- Made a concrete prediction on a hypothetical 2BHK Mumbai flat.